Use ai para ver como clipear outliers

In [173]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [174]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

In [175]:
x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

In [176]:
no_scale_cols = ["Latitude", "Longitude"]
cols = [c for c in x.columns if c not in no_scale_cols]


Q1 = x_train[cols].quantile(0.25)
Q3 = x_train[cols].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

def cap_outliers(df):
    df = df.copy()
    for c in cols:
        df[c] = df[c].clip(lower[c], upper[c])
    return df

x_train = cap_outliers(x_train)
x_val = cap_outliers(x_val)
x_test = cap_outliers(x_test)

mean = x_train[cols].mean()
std = x_train[cols].std()

def scale(df):
    df = df.copy()
    df[cols] = (df[cols] - mean) / std
    return df

x_train = scale(x_train)
x_val = scale(x_val)
x_test = scale(x_test)

In [177]:
x_train = torch.tensor(x_train.values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

x_val = torch.tensor(x_val.values, dtype=torch.float32)
y_val = torch.tensor(y_val.values, dtype=torch.float32).view(-1,1)

x_test = torch.tensor(x_test.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [178]:
class MLP(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(8, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):
        return self.model(x)

In [179]:
def evaluate(model, x, y):
    model.eval()
    with torch.no_grad():
        pred = model(x)
        mse = torch.mean((pred - y)**2).item()
        rmse = np.sqrt(mse)
    return mse, rmse

In [180]:
def train(model, x_train, y_train, x_val, y_val, epochs=50, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()

        pred = model(x_train)
        loss = criterion(pred, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        val_mse, val_rmse = evaluate(model, x_val, y_val)

        print(f"Epoch {epoch+1} | Train Loss: {loss.item():.4f} | Val RMSE: {val_rmse:.4f}")

In [181]:
model1 = MLP()
train(model1, x_train, y_train, x_val, y_val, epochs=12, lr=0.001)

model2 = MLP()
train(model2, x_train, y_train, x_val, y_val, epochs=12, lr=0.0005)

model3 = MLP(hidden=32)
train(model3, x_train, y_train, x_val, y_val, epochs=12, lr=0.001)

Epoch 1 | Train Loss: 8.4765 | Val RMSE: 1.5953
Epoch 2 | Train Loss: 2.5745 | Val RMSE: 1.1510
Epoch 3 | Train Loss: 1.3560 | Val RMSE: 1.6866
Epoch 4 | Train Loss: 2.8791 | Val RMSE: 1.9776
Epoch 5 | Train Loss: 3.9471 | Val RMSE: 1.8818
Epoch 6 | Train Loss: 3.5768 | Val RMSE: 1.5726
Epoch 7 | Train Loss: 2.5070 | Val RMSE: 1.2468
Epoch 8 | Train Loss: 1.5865 | Val RMSE: 1.1130
Epoch 9 | Train Loss: 1.2692 | Val RMSE: 1.2271
Epoch 10 | Train Loss: 1.5356 | Val RMSE: 1.4117
Epoch 11 | Train Loss: 2.0223 | Val RMSE: 1.5156
Epoch 12 | Train Loss: 2.3264 | Val RMSE: 1.4969
Epoch 1 | Train Loss: 6.3146 | Val RMSE: 1.7759
Epoch 2 | Train Loss: 3.1943 | Val RMSE: 1.2425
Epoch 3 | Train Loss: 1.5771 | Val RMSE: 1.0992
Epoch 4 | Train Loss: 1.2368 | Val RMSE: 1.2950
Epoch 5 | Train Loss: 1.7032 | Val RMSE: 1.5100
Epoch 6 | Train Loss: 2.3053 | Val RMSE: 1.6006
Epoch 7 | Train Loss: 2.5869 | Val RMSE: 1.5654
Epoch 8 | Train Loss: 2.4755 | Val RMSE: 1.4457
Epoch 9 | Train Loss: 2.1153 | Val RM

In [182]:
print("Modelo 1:", evaluate(model1, x_test, y_test))
print("Modelo 2:", evaluate(model2, x_test, y_test))
print("Modelo 3:", evaluate(model3, x_test, y_test))

Modelo 1: (2.186199426651001, 1.478580206363862)
Modelo 2: (1.2488512992858887, 1.1175201560982642)
Modelo 3: (1.497917890548706, 1.2238945585910193)
